# 06 – Trino

> Runs against the catalog from `CATALOG` (Lakekeeper, Polaris, RustFS or Unity; default Lakekeeper). What a catalog cannot do is shown by the cell instead of aborting.

Start: `docker compose --profile trino up -d trino` (in the folder `.devcontainer`).

Trino catalogs are fixed files (`trino/catalog/*.properties`), one per catalog: `lk_sts`
(Lakekeeper), `pol_sts` (Polaris), `rfs_sts` (RustFS), `uc_sts` (Unity) – with vended credentials,
on RustFS with its own keys (SigV4). Lakekeeper additionally has `lk_signing` → `demo-signing` with
static RustFS keys (Trino cannot do remote signing).

> Note: PyIceberg cannot read Trino's position delete files (after `UPDATE`/`DELETE` on v2).

In [ ]:
from iceberg_lab.config import PROFILE, REMOTE_SIGNING, SETTINGS, WAREHOUSE_SIGNING, WAREHOUSE_STS, WAREHOUSES
from iceberg_lab.notebook import BUCKET, TRINO_SIGNING, TRINO_STS, catalog_limit, drop_table, header

CATALOG_URI = SETTINGS.catalog_uri
S3_ENDPOINT = SETTINGS.s3_endpoint
header()

In [ ]:
import pandas as pd
import trino

conn = trino.dbapi.connect(host=SETTINGS.trino_host, port=SETTINGS.trino_port, user="notebook", catalog=TRINO_STS)

def sql(q):
    cur = conn.cursor()
    cur.execute(q)
    rows = cur.fetchall()
    return pd.DataFrame(rows, columns=[c[0] for c in cur.description]) if cur.description else rows

sql("SHOW CATALOGS")

In [ ]:
sql("CREATE SCHEMA IF NOT EXISTS nb_trino")
import uuid

ORDERS_NAME = "orders"
with catalog_limit("DROP TABLE"):
    ORDERS_NAME = None
    sql("DROP TABLE IF EXISTS nb_trino.orders")
    ORDERS_NAME = "orders"
if ORDERS_NAME is None:  # e.g. RustFS: Trino cannot drop tables - new name
    ORDERS_NAME = f"orders_{uuid.uuid4().hex[:6]}"
ORDERS = f"nb_trino.{ORDERS_NAME}"
sql(f"""
    CREATE TABLE {ORDERS}
    WITH (partitioning = ARRAY['month(order_date)'])
    AS SELECT CAST(x AS BIGINT) AS order_id,
              CAST(x % 100 AS INTEGER) AS customer_id,
              element_at(ARRAY['open', 'paid', 'shipped'], x % 3 + 1) AS status,
              CAST(x * 1.5 AS DOUBLE) AS amount,
              date_add('day', x % 90, DATE '2026-08-01') AS order_date
       FROM UNNEST(sequence(1, 5000)) AS t(x)
""")
sql(f"SELECT status, count(*) AS n, round(sum(amount), 2) AS total FROM {ORDERS} GROUP BY 1 ORDER BY 1")

## UPDATE, DELETE, MERGE

In [ ]:
sql(f"UPDATE {ORDERS} SET status = 'cancelled' WHERE customer_id = 7")
sql(f"DELETE FROM {ORDERS} WHERE status = 'open' AND order_date < DATE '2026-08-15'")
sql(f"""
    MERGE INTO {ORDERS} t
    USING (VALUES (BIGINT '1', 'refunded'), (BIGINT '999999', 'open')) AS s(order_id, status)
    ON t.order_id = s.order_id
    WHEN MATCHED THEN UPDATE SET status = s.status
    WHEN NOT MATCHED THEN INSERT (order_id, customer_id, status, amount, order_date)
                          VALUES (s.order_id, 0, s.status, 0, current_date)
""")
sql(f"SELECT status, count(*) FROM {ORDERS} GROUP BY 1 ORDER BY 1")

## Metadata tables, time travel, maintenance

In [ ]:
snaps = sql(f'SELECT committed_at, snapshot_id, operation FROM nb_trino."{ORDERS_NAME}$snapshots" ORDER BY committed_at')
snaps

In [ ]:
first = int(snaps.snapshot_id.iloc[1])  # [0] is the empty CREATE snapshot
sql(f"SELECT count(*) AS rows_then FROM {ORDERS} FOR VERSION AS OF {first}")

In [ ]:
sql(f'SELECT content, count(*) AS files FROM nb_trino."{ORDERS_NAME}$files" GROUP BY 1')

In [ ]:
sql(f"ALTER TABLE {ORDERS} EXECUTE optimize")
sql(f"ALTER TABLE {ORDERS} EXECUTE expire_snapshots(retention_threshold => '0s')")
sql(f'SELECT content, count(*) AS files FROM nb_trino."{ORDERS_NAME}$files" GROUP BY 1')

## Join across two warehouses (Lakekeeper only; otherwise within one warehouse)

In [ ]:
other = TRINO_SIGNING or TRINO_STS  # second warehouse only where the catalog offers remote signing
sql(f"CREATE SCHEMA IF NOT EXISTS {other}.nb_trino")
CUSTOMERS = f"{other}.nb_trino.customers_{uuid.uuid4().hex[:6]}"  # fresh name: not every catalog can drop
sql(f"CREATE TABLE {CUSTOMERS} AS SELECT DISTINCT customer_id, 'Customer ' || CAST(customer_id AS VARCHAR) AS name FROM {TRINO_STS}.{ORDERS}")
sql(f"""
    SELECT c.name, count(*) AS orders
    FROM {TRINO_STS}.{ORDERS} o JOIN {CUSTOMERS} c USING (customer_id)
    GROUP BY 1 ORDER BY 2 DESC LIMIT 5
""")